In [19]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

df = pd.read_csv("ecommerce_sales_dataset.csv")

def practical_discount_policy(df, 
                              total_budget=1000000, 
                              min_margin_rate=0.05, 
                              max_discount=0.30,
                              strategy="revenue"):

    df = df.copy()

    # 1) Tính unit_cost
    df["unit_cost"] = (
        df["total_amount"] - df["profit_margin"] - df["shipping_cost"]
    ) / df["quantity"].clip(1)

    # 2) Aggregate theo category
    segments = (
        df.groupby("category")
        .apply(lambda g: pd.Series({
            "avg_price": g["price"].mean(),
            "total_qty": g["quantity"].sum(),
            "avg_cost": np.average(g["unit_cost"], weights=g["quantity"])
        }))
        .reset_index()
    )

    segments["base_margin"] = (
        segments["avg_price"] - segments["avg_cost"]
    ) / segments["avg_price"].clip(0.01)

    # 3) Elasticity
    elasticity_map = {
        "Fashion":     -1.9,
        "Electronics": -1.7,
        "Sports":      -1.6,
        "Home":        -1.5,
        "Beauty":      -1.4,
        "Toys":        -1.4,
        "Grocery":     -1.0
    }
    segments["elasticity"] = segments["category"].map(elasticity_map).fillna(-1.5)

    P0 = segments["avg_price"].to_numpy()
    Q0 = segments["total_qty"].to_numpy()
    C0 = segments["avg_cost"].to_numpy()
    E = segments["elasticity"].to_numpy()
    BM = segments["base_margin"].to_numpy()

    # 4) Upper bounds
    effective_min_margin = np.maximum(min_margin_rate, BM * 0.5)
    max_d_allowed = 1 - (C0 / (P0 * (1 - effective_min_margin) + 1e-9))
    upper_bounds = np.minimum(max_discount, np.maximum(0, max_d_allowed))
    upper_bounds = np.where(BM <= effective_min_margin, 0, upper_bounds)

    bounds = [(0.0, float(ub)) for ub in upper_bounds]

    # 5) Functions
    def new_quantity(d):
        return Q0 * np.power(np.clip(1 - d, 0.01, 1), E)

    def objective(d):
        new_q = new_quantity(d)
        new_price = P0 * (1 - d)

        if strategy == "profit":
            profit = (new_price - C0) * new_q
            return -profit.sum() / 1e6
        else:
            revenue = new_price * new_q
            return -revenue.sum() / 1e6

    def budget_used(d):
        return np.sum(P0 * d * new_quantity(d))

    # 6) Constraint
    constraints = [
        {"type": "ineq", "fun": lambda d: (total_budget - budget_used(d)) / 1e6}
    ]

    # 7) Initial guess
    x0 = np.clip(0.08 * np.abs(E), 0.0, 0.25)

    # 8) Optimize
    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-8, "disp": False}
    )

    if not res.success:
        print(f"Optimizer warning: {res.message}")

    d_opt = np.clip(res.x, 0.0, upper_bounds)

    # 9) Tính kết quả cuối
    final_price = P0 * (1 - d_opt)
    final_qty = new_quantity(d_opt)
    final_revenue = final_price * final_qty
    final_profit = (final_price - C0) * final_qty
    final_margin_rate = np.where(final_price > 0, (final_price - C0) / final_price, 0)

    # Baseline
    baseline_revenue = P0 * Q0
    baseline_profit = (P0 - C0) * Q0

    # 10) Output
    result = segments[["category", "elasticity", "base_margin"]].copy()
    result["Giá Gốc (EUR)"] = np.round(P0, 2)
    result["Giá Vốn (EUR)"] = np.round(C0, 2)
    result["Discount (%)"] = np.round(d_opt * 100, 2)
    result["Doanh Thu Dự Tính"] = np.round(final_revenue, 0)
    result["Lợi Nhuận Dự Tính"] = np.round(final_profit, 0)
    result["Margin Sau Giảm (%)"] = np.round(final_margin_rate * 100, 2)

    # Tổng kết
    total_rev = final_revenue.sum()
    total_profit = final_profit.sum()
    baseline_total_profit = baseline_profit.sum()
    profit_delta = total_profit - baseline_total_profit
    budget_spent = budget_used(d_opt)

    print(f"\n=== KẾT QUẢ TỐI ƯU ({strategy.upper()}) ===")
    print(f"Ngân sách sử dụng: {budget_spent:,.0f} / {total_budget:,.0f} ({budget_spent/total_budget*100:.1f}%)")
    print(f"Doanh thu dự tính: {total_rev:,.0f} EUR")
    print(f"Lợi nhuận baseline: {baseline_total_profit:,.0f} EUR")
    print(f"Lợi nhuận tối ưu: {total_profit:,.0f} EUR")
    print(f"Chênh lệch lợi nhuận vs baseline: {profit_delta:,.0f} EUR ({profit_delta / max(baseline_total_profit, 1e-9) * 100:.2f}%)")

    return result.sort_values("Doanh Thu Dự Tính", ascending=False).reset_index(drop=True)


# ====================== CÁCH SỬ DỤNG ======================

policy_revenue = practical_discount_policy(
    df,
    total_budget=1000000,
    min_margin_rate=0.05,
    max_discount=0.30,
    strategy="revenue"
)
print(policy_revenue.to_string(index=False))



=== KẾT QUẢ TỐI ƯU (REVENUE) ===
Ngân sách sử dụng: 1,000,000 / 1,000,000 (100.0%)
Doanh thu dự tính: 6,764,605 EUR
Lợi nhuận baseline: 1,499,097 EUR
Lợi nhuận tối ưu: 934,655 EUR
Chênh lệch lợi nhuận vs baseline: -564,442 EUR (-37.65%)
   category  elasticity  base_margin  Giá Gốc (EUR)  Giá Vốn (EUR)  Discount (%)  Doanh Thu Dự Tính  Lợi Nhuận Dự Tính  Margin Sau Giảm (%)
Electronics        -1.7     0.161699         374.42         313.88          8.80          3731080.0           301656.0                 8.08
       Home        -1.5     0.310659         137.48          94.77         16.13          1235334.0           220052.0                17.81
     Sports        -1.6     0.339790         107.57          71.02         20.47           770616.0           130924.0                16.99
    Fashion        -1.9     0.383004          54.59          33.68         23.69           636904.0           121968.0                19.15
     Beauty        -1.4     0.479725          26.47          1

C:\Users\USER\AppData\Local\Temp\ipykernel_22692\2663595368.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [20]:
# ====================== MONTE CARLO SIMULATION + PHÂN TÍCH ======================
def run_simulation_and_analysis(df, policy_result, strategy_name="Lợi nhuận", 
                                n_simulations=5000, random_seed=42):
    
    np.random.seed(random_seed)
    
    categories = policy_result['category'].values
    d_opt = policy_result['Discount (%)'].values / 100.0
    P0 = policy_result['Giá Gốc (EUR)'].values
    C0_base = policy_result['Giá Vốn (EUR)'].values
    E_base = policy_result['elasticity'].values

    df_agg = df.groupby('category').agg(total_qty=('quantity', 'sum')).reset_index()
    Q0 = np.array([df_agg[df_agg['category'] == cat]['total_qty'].values[0] 
                   if cat in df_agg['category'].values else 0 for cat in categories])

    profits = []
    revenues = []
    budgets = []

    for _ in range(n_simulations):
        E_sim = E_base * (1 + np.random.normal(0, 0.18, len(E_base)))
        Q_sim = Q0 * (1 + np.random.normal(0, 0.12, len(Q0)))
        C_sim = C0_base * (1 + np.random.normal(0, 0.06, len(C0_base)))

        new_q = Q_sim * np.power(np.clip(1 - d_opt, 0.01, 1), E_sim)
        final_price = P0 * (1 - d_opt)

        rev = (final_price * new_q).sum()
        profit = ((final_price - C_sim) * new_q).sum()
        budget_spent = (P0 * d_opt * new_q).sum()

        profits.append(profit)
        revenues.append(rev)
        budgets.append(budget_spent)

    sim_df = pd.DataFrame({
        'Revenue': revenues,
        'Profit': profits,
        'Budget_Spent': budgets
    })

    # ====================== TÍNH TOÁN CÁC CHỈ SỐ QUAN TRỌNG ======================
    mean_profit = sim_df['Profit'].mean()
    std_profit = sim_df['Profit'].std()
    percentiles = sim_df['Profit'].quantile([0.05, 0.25, 0.5, 0.75, 0.95])

    # Xác suất một số mục tiêu
    prob_loss = (sim_df['Profit'] < 0).mean() * 100
    prob_low_margin = (sim_df['Profit'] / (sim_df['Revenue'] + 1e-9) < 0.05).mean() * 100
    prob_over_budget = (sim_df['Budget_Spent'] > 1000000 * 1.05).mean() * 100

    print(f"\n=== PHÂN TÍCH RỦI RO - {strategy_name.upper()} ===")
    print(f"Số lần mô phỏng          : {n_simulations:,}")
    print(f"Giá trị kỳ vọng (Mean)   : {mean_profit:,.0f} EUR")
    print(f"Độ lệch chuẩn (Std)      : {std_profit:,.0f} EUR")
    print(f"\nPhân vị:")
    print(f"  5%   (VaR 5%)          : {percentiles[0.05]:,.0f} EUR")
    print(f"  25%                     : {percentiles[0.25]:,.0f} EUR")
    print(f"  50%  (Median)           : {percentiles[0.5]:,.0f} EUR")
    print(f"  75%                     : {percentiles[0.75]:,.0f} EUR")
    print(f"  95%                     : {percentiles[0.95]:,.0f} EUR")
    
    print(f"\nXác suất:")
    print(f"  Lỗ vốn (Profit < 0)     : {prob_loss:.2f}%")
    print(f"  Margin < 5%             : {prob_low_margin:.2f}%")
    print(f"  Vượt ngân sách >5%      : {prob_over_budget:.2f}%")

    

# ====================== CHẠY PHÂN TÍCH ======================

print("\n" + "="*80)
print("=== SIMULATION CHO CHIẾN LƯỢC TỐI ƯU DOANH THU ===")
sim_revenue = run_simulation_and_analysis(df, policy_revenue, strategy_name="Doanh thu", n_simulations=5000)


=== SIMULATION CHO CHIẾN LƯỢC TỐI ƯU DOANH THU ===

=== PHÂN TÍCH RỦI RO - DOANH THU ===
Số lần mô phỏng          : 5,000
Giá trị kỳ vọng (Mean)   : 936,812 EUR
Độ lệch chuẩn (Std)      : 230,389 EUR

Phân vị:
  5%   (VaR 5%)          : 567,754 EUR
  25%                     : 785,838 EUR
  50%  (Median)           : 931,850 EUR
  75%                     : 1,087,478 EUR
  95%                     : 1,315,308 EUR

Xác suất:
  Lỗ vốn (Profit < 0)     : 0.02%
  Margin < 5%             : 0.56%
  Vượt ngân sách >5%      : 23.80%


In [21]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

df = pd.read_csv("ecommerce_sales_dataset.csv")

def practical_discount_policy(
    df,
    total_budget=1_000_000,
    min_margin_rate=0.05,
    max_discount=0.30,
    strategy="revenue"
):
    df = df.copy()

    # 1) Tính unit cost
    df["unit_cost"] = (
        df["total_amount"] - df["profit_margin"] - df["shipping_cost"]
    ) / df["quantity"].clip(lower=1)

    # 2) Gom nhóm theo category
    segments = (
        df.groupby("category")
        .apply(lambda g: pd.Series({
            "avg_price": g["price"].mean(),
            "total_qty": g["quantity"].sum(),
            "avg_cost": np.average(g["unit_cost"], weights=g["quantity"])
        }))
        .reset_index()
    )

    segments["base_margin"] = (
        segments["avg_price"] - segments["avg_cost"]
    ) / segments["avg_price"].clip(lower=0.01)

    # 3) Elasticity
    elasticity_map = {
        "Fashion":     -2.9,
        "Electronics": -2.7,
        "Sports":      -2.6,
        "Home":        -2.5,
        "Beauty":      -2.4,
        "Toys":        -2.5,
        "Grocery":     -1.8,
    }
    segments["elasticity"] = segments["category"].map(elasticity_map).fillna(-1.5)

    P0 = segments["avg_price"].to_numpy()
    Q0 = segments["total_qty"].to_numpy()
    C0 = segments["avg_cost"].to_numpy()
    E = segments["elasticity"].to_numpy()
    BM = segments["base_margin"].to_numpy()

    # 4) Giới hạn discount
    effective_min_margin = np.maximum(min_margin_rate, BM * 0.5)
    max_d_allowed = 1 - (C0 / (P0 * (1 - effective_min_margin) + 1e-9))
    upper_bounds = np.minimum(max_discount, np.maximum(0, max_d_allowed))
    upper_bounds = np.where(BM <= effective_min_margin, 0, upper_bounds)

    bounds = [(0.0, float(ub)) for ub in upper_bounds]

    # 5) Hàm phụ
    def new_quantity(d):
        return Q0 * np.power(np.clip(1 - d, 0.01, 1.0), E)

    def revenue_total(d):
        q = new_quantity(d)
        p = P0 * (1 - d)
        return np.sum(p * q)

    def profit_total(d):
        q = new_quantity(d)
        p = P0 * (1 - d)
        return np.sum((p - C0) * q)

    def budget_used(d):
        return np.sum(P0 * d * new_quantity(d))

    baseline_revenue = np.sum(P0 * Q0)
    baseline_profit = np.sum((P0 - C0) * Q0)

    # 6) Objective
    def objective(d):
        if strategy == "profit":
            return -profit_total(d) / 1e6
        return -revenue_total(d) / 1e6

    # 7) Constraints
    # budget_used <= total_budget
    # profit_total >= baseline_profit
    constraints = [
        {"type": "ineq", "fun": lambda d: (total_budget - budget_used(d)) / 1e6},
        {"type": "ineq", "fun": lambda d: (profit_total(d) - baseline_profit) / 1e6},
    ]

    # 8) Initial guess
    x0 = np.minimum(np.clip(0.08 * np.abs(E), 0.0, 0.25), upper_bounds)

    # Nếu x0 làm profit < baseline thì bắt đầu từ 0% discount để luôn feasible
    if profit_total(x0) < baseline_profit:
        x0 = np.zeros_like(P0)

    # 9) Optimize
    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-8, "disp": False}
    )

    if not res.success:
        print(f"Optimizer warning: {res.message}")

    d_opt = np.clip(res.x, 0.0, upper_bounds)

    # Fallback an toàn: nếu nghiệm tối ưu vẫn thua baseline thì quay về no-discount
    if profit_total(d_opt) < baseline_profit:
        d_opt = np.zeros_like(d_opt)

    # 10) Kết quả cuối
    final_price = P0 * (1 - d_opt)
    final_qty = new_quantity(d_opt)
    final_revenue = final_price * final_qty
    final_profit = (final_price - C0) * final_qty
    final_margin_rate = np.where(final_price > 0, (final_price - C0) / final_price, 0)

    result = segments[["category", "elasticity", "base_margin"]].copy()
    result["Giá Gốc (EUR)"] = np.round(P0, 2)
    result["Giá Vốn (EUR)"] = np.round(C0, 2)
    result["Discount (%)"] = np.round(d_opt * 100, 2)
    result["Doanh Thu Dự Tính"] = np.round(final_revenue, 0)
    result["Lợi Nhuận Dự Tính"] = np.round(final_profit, 0)
    result["Margin Sau Giảm (%)"] = np.round(final_margin_rate * 100, 2)

    total_rev = final_revenue.sum()
    total_profit = final_profit.sum()
    profit_delta = total_profit - baseline_profit
    budget_spent = budget_used(d_opt)

    print(f"\n=== KẾT QUẢ TỐI ƯU ({strategy.upper()}) ===")
    print(f"Ngân sách sử dụng: {budget_spent:,.0f} / {total_budget:,.0f} ({budget_spent / total_budget * 100:.1f}%)")
    print(f"Doanh thu baseline: {baseline_revenue:,.0f} EUR")
    print(f"Doanh thu tối ưu: {total_rev:,.0f} EUR")
    print(f"Lợi nhuận baseline: {baseline_profit:,.0f} EUR")
    print(f"Lợi nhuận tối ưu: {total_profit:,.0f} EUR")
    print(f"Chênh lệch lợi nhuận vs baseline: {profit_delta:,.0f} EUR ({profit_delta / max(baseline_profit, 1e-9) * 100:.2f}%)")

    return result.sort_values("Doanh Thu Dự Tính", ascending=False).reset_index(drop=True)


# ====================== CÁCH SỬ DỤNG ======================

policy_revenue = practical_discount_policy(
    df,
    total_budget=1_000_000,
    min_margin_rate=0.05,
    max_discount=0.30,
    strategy="revenue"   # tối đa revenue nhưng vẫn ép profit >= baseline
)
print(policy_revenue.to_string(index=False))



=== KẾT QUẢ TỐI ƯU (REVENUE) ===
Ngân sách sử dụng: 0 / 1,000,000 (0.0%)
Doanh thu baseline: 6,188,646 EUR
Doanh thu tối ưu: 6,188,646 EUR
Lợi nhuận baseline: 1,499,097 EUR
Lợi nhuận tối ưu: 1,499,097 EUR
Chênh lệch lợi nhuận vs baseline: 0 EUR (0.00%)
   category  elasticity  base_margin  Giá Gốc (EUR)  Giá Vốn (EUR)  Discount (%)  Doanh Thu Dự Tính  Lợi Nhuận Dự Tính  Margin Sau Giảm (%)
Electronics        -2.7     0.161699         374.42         313.88           0.0          3498194.0           565654.0                16.17
       Home        -2.5     0.310659         137.48          94.77           0.0          1131359.0           351467.0                31.07
     Sports        -2.6     0.339790         107.57          71.02           0.0           671688.0           228233.0                33.98
    Fashion        -2.9     0.383004          54.59          33.68           0.0           499364.0           191258.0                38.30
     Beauty        -2.4     0.479725          

C:\Users\USER\AppData\Local\Temp\ipykernel_22692\2550116858.py:24: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [22]:
# ====================== MONTE CARLO SIMULATION + PHÂN TÍCH ======================
def run_simulation_and_analysis(df, policy_result, strategy_name="Lợi nhuận", 
                                n_simulations=5000, random_seed=42):
    
    np.random.seed(random_seed)
    
    categories = policy_result['category'].values
    d_opt = policy_result['Discount (%)'].values / 100.0
    P0 = policy_result['Giá Gốc (EUR)'].values
    C0_base = policy_result['Giá Vốn (EUR)'].values
    E_base = policy_result['elasticity'].values

    df_agg = df.groupby('category').agg(total_qty=('quantity', 'sum')).reset_index()
    Q0 = np.array([df_agg[df_agg['category'] == cat]['total_qty'].values[0] 
                   if cat in df_agg['category'].values else 0 for cat in categories])

    profits = []
    revenues = []
    budgets = []

    for _ in range(n_simulations):
        E_sim = E_base * (1 + np.random.normal(0, 0.18, len(E_base)))
        Q_sim = Q0 * (1 + np.random.normal(0, 0.12, len(Q0)))
        C_sim = C0_base * (1 + np.random.normal(0, 0.06, len(C0_base)))

        new_q = Q_sim * np.power(np.clip(1 - d_opt, 0.01, 1), E_sim)
        final_price = P0 * (1 - d_opt)

        rev = (final_price * new_q).sum()
        profit = ((final_price - C_sim) * new_q).sum()
        budget_spent = (P0 * d_opt * new_q).sum()

        profits.append(profit)
        revenues.append(rev)
        budgets.append(budget_spent)

    sim_df = pd.DataFrame({
        'Revenue': revenues,
        'Profit': profits,
        'Budget_Spent': budgets
    })

    # ====================== TÍNH TOÁN CÁC CHỈ SỐ QUAN TRỌNG ======================
    mean_profit = sim_df['Profit'].mean()
    std_profit = sim_df['Profit'].std()
    percentiles = sim_df['Profit'].quantile([0.05, 0.25, 0.5, 0.75, 0.95])

    # Xác suất một số mục tiêu
    prob_loss = (sim_df['Profit'] < 0).mean() * 100
    prob_low_margin = (sim_df['Profit'] / (sim_df['Revenue'] + 1e-9) < 0.05).mean() * 100
    prob_over_budget = (sim_df['Budget_Spent'] > 1000000 * 1.05).mean() * 100

    print(f"\n=== PHÂN TÍCH RỦI RO - {strategy_name.upper()} ===")
    print(f"Số lần mô phỏng          : {n_simulations:,}")
    print(f"Giá trị kỳ vọng (Mean)   : {mean_profit:,.0f} EUR")
    print(f"Độ lệch chuẩn (Std)      : {std_profit:,.0f} EUR")
    print(f"\nPhân vị:")
    print(f"  5%   (VaR 5%)          : {percentiles[0.05]:,.0f} EUR")
    print(f"  25%                     : {percentiles[0.25]:,.0f} EUR")
    print(f"  50%  (Median)           : {percentiles[0.5]:,.0f} EUR")
    print(f"  75%                     : {percentiles[0.75]:,.0f} EUR")
    print(f"  95%                     : {percentiles[0.95]:,.0f} EUR")
    
    print(f"\nXác suất:")
    print(f"  Lỗ vốn (Profit < 0)     : {prob_loss:.2f}%")
    print(f"  Margin < 5%             : {prob_low_margin:.2f}%")
    print(f"  Vượt ngân sách >5%      : {prob_over_budget:.2f}%")

    

# ====================== CHẠY PHÂN TÍCH ======================

print("\n" + "="*80)
print("=== SIMULATION CHO CHIẾN LƯỢC TỐI ƯU DOANH THU ===")
sim_revenue = run_simulation_and_analysis(df, policy_revenue, strategy_name="Doanh thu", n_simulations=5000)


=== SIMULATION CHO CHIẾN LƯỢC TỐI ƯU DOANH THU ===

=== PHÂN TÍCH RỦI RO - DOANH THU ===
Số lần mô phỏng          : 5,000
Giá trị kỳ vọng (Mean)   : 1,500,361 EUR
Độ lệch chuẩn (Std)      : 207,219 EUR

Phân vị:
  5%   (VaR 5%)          : 1,173,624 EUR
  25%                     : 1,362,042 EUR
  50%  (Median)           : 1,495,363 EUR
  75%                     : 1,640,294 EUR
  95%                     : 1,848,930 EUR

Xác suất:
  Lỗ vốn (Profit < 0)     : 0.00%
  Margin < 5%             : 0.00%
  Vượt ngân sách >5%      : 0.00%
